In [1]:
from pathlib import Path
import os, sys, subprocess, json, time
ROOT=Path('/tmp/e12_source'); OUT=Path('/kaggle/working')
SOURCE='e1f0c9861fcdd38d07cd24e746900d53a314b740'
env={**os.environ,'UV_PYTHON_INSTALL_DIR':'/tmp/e12_python','UV_CACHE_DIR':'/tmp/e12_cache','UV_NATIVE_TLS':'true','CUBLAS_WORKSPACE_CONFIG':':4096:8','OMP_NUM_THREADS':'1','MPLCONFIGDIR':'/tmp/e12_mpl','MPLBACKEND':'Agg','NANO_SOURCE_COMMIT':SOURCE,'NANO_SESSION_LIMIT_SECONDS':'43200'}
def command(args,cwd=None):
    print('EXEC',args,flush=True)
    with (OUT/'bootstrap.log').open('a') as log:
        result=subprocess.run(args,cwd=cwd,env=env,stdout=log,stderr=subprocess.STDOUT)
    if result.returncode:
        print((OUT/'bootstrap.log').read_text()[-4000:],flush=True)
        raise RuntimeError('E12 gate failed; preserve output')
assert not ROOT.exists()
command(['git','clone','https://github.com/rayyan-merchant/Nano-Language-Model.git',str(ROOT)])
command(['git','checkout','--detach',SOURCE],ROOT)
assert not subprocess.check_output(['git','status','--porcelain'],cwd=ROOT,text=True).strip()
command([sys.executable,'-m','pip','install','--target','/tmp/e12_tools','uv==0.12.7'])
UV=Path('/tmp/e12_tools/bin/uv')
command([str(UV),'python','install','3.11.14'])
command([str(UV),'venv','--seed','--python','3.11.14','/tmp/e12_venv'])
PY=Path('/tmp/e12_venv/bin/python')
freeze=(ROOT/'docs/qualifications/evidence/rayyan_requirements_freeze.txt').read_text().splitlines()
lock=OUT/'qualified-lock.txt';lock.write_text(chr(10).join(x for x in freeze if not x.startswith('-e '))+chr(10))
command([str(UV),'pip','install','--python',str(PY),'torch==2.5.1+cu124','--index-url','https://download.pytorch.org/whl/cu124'])
command([str(UV),'pip','install','--python',str(PY),'-r',str(lock)])
command([str(UV),'pip','install','--python',str(PY),'-e',str(ROOT),'--no-deps'])
command([str(PY),'-m','pip','check'])
from shutil import copytree, copyfile
original=OUT/'verified_final_original'; (original/'snapshots').mkdir(parents=True)
candidates=[]
for path in Path('/kaggle/input').rglob('export_manifest.json'):
    if 'research_snapshots' in path.parts:
        m=json.loads(path.read_text())
        if m['step']==5000: candidates.append((len(m['files']),path))
assert candidates, 'No durable step5000 snapshot mounted'
selected=max(candidates,key=lambda item:item[0])[1].parent
copytree(selected,original/'snapshots'/selected.name)
completions=[]
for path in Path('/kaggle/input').rglob('completion.json'):
    if path.parent.name=='research':
        c=json.loads(path.read_text())
        if c['run_id']=='E12_seed42_4fef43b3dd454ee0bda3a63d52280c8b': completions.append(path)
assert len(completions)==1
copyfile(completions[0],original/'completion.json')
env['PYTHONPATH']=str(ROOT/'scripts/pilot')
code="from pathlib import Path; from recover import recover; recover(Path('/tmp/e12_source'),Path('/kaggle/working/fresh_final_recovery'),Path('/kaggle/working/verified_final_original'))"
command([str(PY),'-c',code],ROOT)
print((OUT/'fresh_final_recovery/recovery.json').read_text(),flush=True)


EXEC ['git', 'clone', 'https://github.com/rayyan-merchant/Nano-Language-Model.git', '/tmp/e12_source']
EXEC ['git', 'checkout', '--detach', 'e1f0c9861fcdd38d07cd24e746900d53a314b740']
EXEC ['/usr/bin/python3', '-m', 'pip', 'install', '--target', '/tmp/e12_tools', 'uv==0.12.7']
EXEC ['/tmp/e12_tools/bin/uv', 'python', 'install', '3.11.14']
EXEC ['/tmp/e12_tools/bin/uv', 'venv', '--seed', '--python', '3.11.14', '/tmp/e12_venv']
EXEC ['/tmp/e12_tools/bin/uv', 'pip', 'install', '--python', '/tmp/e12_venv/bin/python', 'torch==2.5.1+cu124', '--index-url', 'https://download.pytorch.org/whl/cu124']
EXEC ['/tmp/e12_tools/bin/uv', 'pip', 'install', '--python', '/tmp/e12_venv/bin/python', '-r', '/kaggle/working/qualified-lock.txt']
EXEC ['/tmp/e12_tools/bin/uv', 'pip', 'install', '--python', '/tmp/e12_venv/bin/python', '-e', '/tmp/e12_source', '--no-deps']
EXEC ['/tmp/e12_venv/bin/python', '-m', 'pip', 'check']
EXEC ['/tmp/e12_venv/bin/python', '-c', "from pathlib import Path; from recover import